In [19]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import joblib
from typing import Dict, List, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

# ML imports
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (
    roc_auc_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_curve, auc,
    mean_absolute_error, mean_squared_error, r2_score
)

# Optional imports with graceful fallback
try:
    from xgboost import XGBClassifier, XGBRegressor
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("⚠ XGBoost not installed - using RandomForest")

try:
    import shap
    HAS_SHAP = True
except ImportError:
    HAS_SHAP = False
    print("⚠ SHAP not installed - skipping explainability")

⚠ XGBoost not installed - using RandomForest
⚠ SHAP not installed - skipping explainability


In [20]:
class Config:
    """Centralized configuration"""
    # Use the exact path where your file is located
    ROOT = Path("C:/Users/naman/OneDrive/Desktop/retail revenue leakage detector")
    DATA_DIR = ROOT
    OUTPUTS = ROOT / "outputs"
    FIG_DIR = OUTPUTS / "figures"
    MODELS_DIR = OUTPUTS / "models"
    
    # Model parameters
    RANDOM_STATE = 42
    TEST_SIZE = 0.15
    VAL_SIZE = 0.1765  # ~15% of original data
    
    # Business parameters
    INTERVENTION_COST = 50
    INTERVENTION_SUCCESS = 0.4
    TOP_K_PERCENT = 0.05
    
    # Data processing
    LEAK_STATUSES = ['canceled', 'unavailable', 'refunded']
    TIMESTAMP_CANDIDATES = [
        'order_purchase_timestamp', 'order_approved_at',
        'order_delivered_customer_date', 'order_estimated_delivery_date'
    ]
    QUANTITY_CANDIDATES = [
        'quantity', 'product_quantity', 'order_item_id',
        'order_item_sequence', 'payments_number'
    ]
    FIRST_COLS = [
        'order_purchase_timestamp', 'order_estimated_delivery_date',
        'order_delivered_customer_date', 'review_score',
        'payment_type', 'customer_id', 'seller_id', 'order_status'
    ]

    @classmethod
    def setup_directories(cls):
        """Create output directories"""
        for path in [cls.OUTPUTS, cls.FIG_DIR, cls.MODELS_DIR]:
            path.mkdir(parents=True, exist_ok=True)
        print(f"✓ Directories ready: {cls.OUTPUTS}")

In [21]:
class DataLoader:
    """Efficient data loading with optimized dtypes"""
    
    @staticmethod
    def load_cleaned_data(config: Config) -> pd.DataFrame:
        """Load data with optimized memory usage"""
        # Try multiple locations
        possible_paths = [
            Path("C:/Users/naman/OneDrive/Desktop/retail revenue leakage detector/olist_cleaned.csv"),
            config.DATA_DIR / "olist_cleaned.csv",
            config.ROOT / "olist_cleaned.csv"
        ]
        
        clean_file = None
        for path in possible_paths:
            if path.exists():
                clean_file = path
                break
        
        if clean_file is None:
            raise FileNotFoundError(f"Cleaned file not found. Tried: {possible_paths}")
        
        # Optimized loading with dtype inference
        df = pd.read_csv(clean_file, low_memory=False)
        
        # Downcast numeric columns to save memory - O(n)
        for col in df.select_dtypes(include=['float64']).columns:
            df[col] = pd.to_numeric(df[col], downcast='float')
        for col in df.select_dtypes(include=['int64']).columns:
            df[col] = pd.to_numeric(df[col], downcast='integer')
        
        print(f"✓ Loaded: {clean_file.name} | Shape: {df.shape} | Memory: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
        return df


class FeatureEngineer:
    """Optimized feature engineering pipeline"""
    
    def __init__(self, config: Config):
        self.config = config
        self.timestamp_col = None
    
    def find_timestamp_column(self, df: pd.DataFrame) -> Optional[str]:
        """Find best timestamp column - O(k) where k is small"""
        for col in self.config.TIMESTAMP_CANDIDATES:
            if col in df.columns:
                return col
        
        # Fallback search
        for col in df.columns:
            if 'purchase' in col.lower():
                return col
        return None
    
    def process_timestamps(self, df: pd.DataFrame) -> pd.DataFrame:
        """Convert timestamps efficiently - O(n)"""
        self.timestamp_col = self.find_timestamp_column(df)
        
        if self.timestamp_col and self.timestamp_col in df.columns:
            df[self.timestamp_col] = pd.to_datetime(df[self.timestamp_col], errors='coerce')
            print(f"✓ Using timestamp: {self.timestamp_col}")
        else:
            print("⚠ No timestamp found - using random splits")
        
        return df
    
    def create_targets(self, df: pd.DataFrame) -> pd.DataFrame:
        """Create target variables efficiently - O(n)"""
        if 'order_status' not in df.columns:
            raise ValueError("'order_status' column required")
        
        # Vectorized operations instead of row-wise
        df['order_status_norm'] = df['order_status'].str.lower().str.strip()
        df['leakage_flag_row'] = df['order_status_norm'].isin(self.config.LEAK_STATUSES).astype(np.int8)
        
        return df
    
    def handle_quantity(self, df: pd.DataFrame) -> Tuple[pd.DataFrame, str]:
        """Determine quantity column - O(k)"""
        qty_col = None
        
        # Find quantity column
        for col in self.config.QUANTITY_CANDIDATES:
            if col in df.columns:
                qty_col = col
                break
        
        # Set default quantity
        if qty_col is None or not np.issubdtype(df[qty_col].dtype, np.number):
            df['quantity'] = 1
            qty_col = 'quantity'
        
        return df, qty_col
    
    def aggregate_to_order_level(self, df: pd.DataFrame) -> pd.DataFrame:
        """Optimized aggregation - O(n log n) due to groupby"""
        # Handle price column
        if 'price' not in df.columns and 'payment_value' in df.columns:
            df['price'] = df['payment_value']
        
        df, qty_col = self.handle_quantity(df)
        
        # Vectorized revenue calculation - O(n)
        df['row_revenue'] = df['price'].fillna(0) * df[qty_col].fillna(1)
        
        # Single-pass aggregation with multiple operations - O(n log n)
        agg_dict = {
            'row_revenue': 'sum',
            'leakage_flag_row': 'max'
        }
        
        # Add first values for existing columns
        for col in self.config.FIRST_COLS:
            if col in df.columns:
                agg_dict[col] = 'first'
        
        order_df = df.groupby('order_id', as_index=False).agg(agg_dict)
        
        # Rename columns
        order_df.rename(columns={
            'row_revenue': 'order_value',
            'leakage_flag_row': 'leakage_flag'
        }, inplace=True)
        
        # Create loss amount - O(n)
        order_df['loss_amount'] = order_df['order_value'] * order_df['leakage_flag']
        
        print(f"✓ Order-level aggregation: {order_df.shape}")
        return order_df
    
    def engineer_features(self, df: pd.DataFrame) -> pd.DataFrame:
        """Create derived features - O(n)"""
        odf = df.copy()
        
        # Delivery delay - O(n)
        if all(col in odf.columns for col in ['order_estimated_delivery_date', 'order_delivered_customer_date']):
            odf['order_estimated_delivery_date'] = pd.to_datetime(odf['order_estimated_delivery_date'], errors='coerce')
            odf['order_delivered_customer_date'] = pd.to_datetime(odf['order_delivered_customer_date'], errors='coerce')
            odf['delivery_delay'] = (
                odf['order_delivered_customer_date'] - odf['order_estimated_delivery_date']
            ).dt.days.fillna(0).clip(-30, 120)
        else:
            odf['delivery_delay'] = 0
        
        # Temporal features - O(n)
        if self.timestamp_col and self.timestamp_col in odf.columns:
            odf[self.timestamp_col] = pd.to_datetime(odf[self.timestamp_col], errors='coerce')
            odf['order_month'] = odf[self.timestamp_col].dt.to_period('M').astype(str)
            odf['order_weekday'] = odf[self.timestamp_col].dt.weekday.astype(np.int8)
        else:
            odf['order_month'] = 'unknown'
            odf['order_weekday'] = -1
        
        # Review score imputation - O(n)
        if 'review_score' in odf.columns:
            median_score = odf['review_score'].median()
            odf['review_score'] = odf['review_score'].fillna(median_score)
        else:
            odf['review_score'] = 3.0
        
        # Payment type - O(n)
        if 'payment_type' in odf.columns:
            odf['payment_type'] = odf['payment_type'].astype(str).fillna('unknown')
        else:
            odf['payment_type'] = 'unknown'
        
        # Derived features - O(n)
        odf['order_value_log'] = np.log1p(odf['order_value'])
        odf['delay_value_interaction'] = odf['delivery_delay'] * odf['order_value']
        
        print(f"✓ Feature engineering complete: {len(odf.columns)} columns")
        return odf

In [22]:
class DataSplitter:
    """Efficient train/val/test splitting"""
    
    def __init__(self, config: Config):
        self.config = config
    
    def split_data(self, df: pd.DataFrame, timestamp_col: Optional[str]) -> Tuple:
        """Split data with time-based or stratified approach"""
        use_time_split = timestamp_col is not None and timestamp_col in df.columns
        
        if use_time_split:
            return self._time_based_split(df, timestamp_col)
        else:
            return self._stratified_split(df)
    
    def _time_based_split(self, df: pd.DataFrame, ts_col: str) -> Tuple:
        """Time-based split - O(n log n) for sorting"""
        df_sorted = df.sort_values(by=ts_col).reset_index(drop=True)
        n = len(df_sorted)
        
        train_end = int(0.7 * n)
        val_end = int(0.85 * n)
        
        train = df_sorted.iloc[:train_end].copy()
        val = df_sorted.iloc[train_end:val_end].copy()
        test = df_sorted.iloc[val_end:].copy()
        
        print(f"✓ Time-based split: Train={len(train)}, Val={len(val)}, Test={len(test)}")
        return train, val, test
    
    def _stratified_split(self, df: pd.DataFrame) -> Tuple:
        """Stratified split - O(n)"""
        train_val, test = train_test_split(
            df, test_size=self.config.TEST_SIZE,
            random_state=self.config.RANDOM_STATE,
            stratify=df['leakage_flag']
        )
        
        train, val = train_test_split(
            train_val, test_size=self.config.VAL_SIZE,
            random_state=self.config.RANDOM_STATE,
            stratify=train_val['leakage_flag']
        )
        
        print(f"✓ Stratified split: Train={len(train)}, Val={len(val)}, Test={len(test)}")
        return train, val, test


In [23]:
class FeaturePreparer:
    """Prepare features with efficient one-hot encoding"""
    
    def __init__(self, config: Config):
        self.config = config
        self.feature_cols = []
        self.payment_dummies_cols = []
        self.scaler = StandardScaler()
    
    def prepare_features(self, train_df, val_df, test_df) -> Tuple:
        """Prepare all features - O(n * m) where m is feature count"""
        # Make copies to avoid modifying original dataframes
        train_df = train_df.copy()
        val_df = val_df.copy()
        test_df = test_df.copy()
        
        # Base numeric features
        self.feature_cols = [
            'order_value', 'order_value_log', 'delivery_delay',
            'review_score', 'delay_value_interaction'
        ]
        
        # Handle payment type dummies efficiently
        if 'payment_type' in train_df.columns:
            train_df, val_df, test_df = self._add_payment_dummies(train_df, val_df, test_df)
        
        # Fill missing values - O(n * m)
        for df in [train_df, val_df, test_df]:
            for col in self.feature_cols:
                if col not in df.columns:
                    df[col] = 0
            df[self.feature_cols] = df[self.feature_cols].fillna(0)
            
            # CRITICAL: Fill NaN in target variables
            df['leakage_flag'] = df['leakage_flag'].fillna(0).astype(np.int8)
            df['loss_amount'] = df['loss_amount'].fillna(0).astype(np.float32)
        
        # Store cleaned dataframes for later use
        self.train_df_clean = train_df
        self.val_df_clean = val_df
        self.test_df_clean = test_df
        
        # Extract X, y for classification and regression
        X_train_cl = train_df[self.feature_cols].values
        y_train_cl = train_df['leakage_flag'].values.astype(np.int8)
        X_val_cl = val_df[self.feature_cols].values
        y_val_cl = val_df['leakage_flag'].values.astype(np.int8)
        X_test_cl = test_df[self.feature_cols].values
        y_test_cl = test_df['leakage_flag'].values.astype(np.int8)
        
        X_train_rg = train_df[self.feature_cols].values
        y_train_rg = train_df['loss_amount'].values.astype(np.float32)
        X_val_rg = val_df[self.feature_cols].values
        y_val_rg = val_df['loss_amount'].values.astype(np.float32)
        X_test_rg = test_df[self.feature_cols].values
        y_test_rg = test_df['loss_amount'].values.astype(np.float32)
        
        # Scale features - O(n * m)
        X_train_cl_scaled = self.scaler.fit_transform(X_train_cl)
        X_val_cl_scaled = self.scaler.transform(X_val_cl)
        X_test_cl_scaled = self.scaler.transform(X_test_cl)
        
        print(f"✓ Features prepared: {len(self.feature_cols)} features")
        
        return (X_train_cl, y_train_cl, X_val_cl, y_val_cl, X_test_cl, y_test_cl,
                X_train_rg, y_train_rg, X_val_rg, y_val_rg, X_test_rg, y_test_rg,
                X_train_cl_scaled, X_val_cl_scaled, X_test_cl_scaled)
    
    def _add_payment_dummies(self, train_df, val_df, test_df):
        """Efficient one-hot encoding - O(n)"""
        # Create dummies for train and get column names
        train_dummies = pd.get_dummies(train_df['payment_type'], prefix='pay', drop_first=True)
        self.payment_dummies_cols = list(train_dummies.columns)
        self.feature_cols.extend(self.payment_dummies_cols)
        
        # Add dummies to train dataframe
        train_df = pd.concat([train_df.reset_index(drop=True), train_dummies], axis=1)
        
        # Create and add dummies for validation
        val_dummies = pd.get_dummies(val_df['payment_type'], prefix='pay', drop_first=True)
        for col in self.payment_dummies_cols:
            if col not in val_dummies.columns:
                val_dummies[col] = 0
        val_df = pd.concat([val_df.reset_index(drop=True), val_dummies[self.payment_dummies_cols]], axis=1)
        
        # Create and add dummies for test
        test_dummies = pd.get_dummies(test_df['payment_type'], prefix='pay', drop_first=True)
        for col in self.payment_dummies_cols:
            if col not in test_dummies.columns:
                test_dummies[col] = 0
        test_df = pd.concat([test_df.reset_index(drop=True), test_dummies[self.payment_dummies_cols]], axis=1)
        
        return train_df, val_df, test_df

In [24]:
class ModelTrainer:
    """Efficient model training pipeline"""
    
    def __init__(self, config: Config):
        self.config = config
        self.models = {}
    
    def train_classifiers(self, X_train, y_train, X_train_scaled) -> Dict:
        """Train classification models"""
        print("\n🔹 Training classifiers...")
        
        # Logistic Regression
        log_clf = LogisticRegression(
            class_weight='balanced',
            max_iter=1000,
            random_state=self.config.RANDOM_STATE,
            n_jobs=-1  # Parallel processing
        )
        log_clf.fit(X_train_scaled, y_train)
        self.models['logistic'] = log_clf
        
        # Tree-based model (XGBoost or RandomForest)
        if HAS_XGB:
            tree_clf = XGBClassifier(
                n_estimators=200,
                tree_method='hist',  # Faster histogram-based algorithm
                random_state=self.config.RANDOM_STATE,
                n_jobs=-1
            )
        else:
            tree_clf = RandomForestClassifier(
                n_estimators=200,
                class_weight='balanced',
                random_state=self.config.RANDOM_STATE,
                n_jobs=-1
            )
        
        tree_clf.fit(X_train, y_train)
        self.models['tree_clf'] = tree_clf
        
        print("✓ Classifiers trained")
        return self.models
    
    def train_regressors(self, X_train, y_train) -> Dict:
        """Train regression models"""
        print("\n🔹 Training regressors...")
        
        # Linear Regression
        lin_reg = LinearRegression(n_jobs=-1)
        lin_reg.fit(X_train, y_train)
        self.models['linear_reg'] = lin_reg
        
        # Tree-based regressor
        if HAS_XGB:
            tree_reg = XGBRegressor(
                n_estimators=200,
                tree_method='hist',
                random_state=self.config.RANDOM_STATE,
                n_jobs=-1
            )
        else:
            tree_reg = RandomForestRegressor(
                n_estimators=200,
                random_state=self.config.RANDOM_STATE,
                n_jobs=-1
            )
        
        tree_reg.fit(X_train, y_train)
        self.models['tree_reg'] = tree_reg
        
        print("✓ Regressors trained")
        return self.models
    
    def save_models(self, config: Config, scaler):
        """Save all models"""
        joblib.dump(scaler, config.MODELS_DIR / "scaler.pkl")
        for name, model in self.models.items():
            joblib.dump(model, config.MODELS_DIR / f"{name}.pkl")
        print(f"✓ Saved {len(self.models)} models + scaler")

In [25]:
class Evaluator:
    """Efficient model evaluation"""
    
    @staticmethod
    def classification_metrics(y_true, y_pred, y_proba) -> Dict:
        """Calculate classification metrics - O(n)"""
        return {
            'roc_auc': roc_auc_score(y_true, y_proba),
            'precision': precision_score(y_true, y_pred, zero_division=0),
            'recall': recall_score(y_true, y_pred, zero_division=0),
            'f1': f1_score(y_true, y_pred, zero_division=0)
        }
    
    @staticmethod
    def regression_metrics(y_true, y_pred) -> Dict:
        """Calculate regression metrics - O(n)"""
        return {
            'mae': mean_absolute_error(y_true, y_pred),
            'rmse': mean_squared_error(y_true, y_pred, squared=False),
            'r2': r2_score(y_true, y_pred)
        }
    
    @staticmethod
    def precision_at_k(y_true, y_proba, k_frac=0.05) -> float:
        """Calculate precision@k - O(n log n) for sorting"""
        k = max(1, int(k_frac * len(y_proba)))
        topk_idx = np.argsort(y_proba)[-k:]
        return y_true[topk_idx].sum() / k
    
    @staticmethod
    def plot_roc_curve(y_true, y_proba, save_path):
        """Plot ROC curve"""
        fpr, tpr, _ = roc_curve(y_true, y_proba)
        roc_auc = auc(fpr, tpr)
        
        plt.figure(figsize=(6, 4))
        plt.plot(fpr, tpr, label=f"AUC={roc_auc:.3f}", linewidth=2)
        plt.plot([0, 1], [0, 1], '--', color='grey', label='Random')
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.title("ROC Curve (Validation)")
        plt.legend()
        plt.tight_layout()
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()
    
    @staticmethod
    def plot_confusion_matrix(y_true, y_pred, save_path):
        """Plot confusion matrix"""
        cm = confusion_matrix(y_true, y_pred)
        plt.figure(figsize=(6, 4))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
        plt.title("Confusion Matrix (Validation)")
        plt.xlabel("Predicted")
        plt.ylabel("True")
        plt.tight_layout()
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()

In [26]:
class BusinessSimulator:
    """Efficient business impact simulation"""
    
    def __init__(self, config: Config):
        self.config = config
    
    def simulate_threshold(self, test_df, threshold: float) -> Dict:
        """Simulate intervention at given threshold - O(n)"""
        flagged = test_df[test_df['pred_proba'] >= threshold]
        num_flagged = len(flagged)
        
        if num_flagged == 0:
            return {
                'threshold': threshold, 'flagged': 0, 'precision': 0,
                'true_positives': 0, 'prevented': 0, 'revenue_preserved': 0,
                'intervention_cost': 0, 'net': 0
            }
        
        true_positives = flagged['true_leak'].sum()
        prevented = int(true_positives * self.config.INTERVENTION_SUCCESS)
        avg_order_value = flagged['order_value'].mean()
        revenue_preserved = prevented * avg_order_value
        intervention_cost = num_flagged * self.config.INTERVENTION_COST
        net = revenue_preserved - intervention_cost
        precision = true_positives / num_flagged
        
        return {
            'threshold': threshold,
            'flagged': num_flagged,
            'precision': float(precision),
            'true_positives': int(true_positives),
            'prevented': prevented,
            'revenue_preserved': float(revenue_preserved),
            'intervention_cost': float(intervention_cost),
            'net': float(net)
        }
    
    def run_simulation(self, test_df, proba_col='pred_proba') -> pd.DataFrame:
        """Run simulation across multiple thresholds - O(k * n) where k is small"""
        thresholds = np.linspace(0.1, 0.9, 9)
        results = [self.simulate_threshold(test_df, t) for t in thresholds]
        sim_df = pd.DataFrame(results)
        
        print("\n📊 Business Simulation Results:")
        print(sim_df[['threshold', 'flagged', 'precision', 'net']].to_string(index=False))
        
        return sim_df

In [27]:
def main():
    """Main execution pipeline"""
    print("=" * 80)
    print("REVENUE LEAKAGE DETECTION - OPTIMIZED PIPELINE")
    print("=" * 80)
    
    # Setup
    config = Config()
    config.setup_directories()
    
    # Load data
    print("\n📥 Loading data...")
    loader = DataLoader()
    df = loader.load_cleaned_data(config)
    
    # Feature engineering
    print("\n🔧 Engineering features...")
    engineer = FeatureEngineer(config)
    df = engineer.process_timestamps(df)
    df = engineer.create_targets(df)
    order_df = engineer.aggregate_to_order_level(df)
    odf = engineer.engineer_features(order_df)
    
    # Split data
    print("\n✂️ Splitting data...")
    splitter = DataSplitter(config)
    train_df, val_df, test_df = splitter.split_data(odf, engineer.timestamp_col)
    
    # Prepare features
    print("\n🎯 Preparing features...")
    preparer = FeaturePreparer(config)
    (X_train_cl, y_train_cl, X_val_cl, y_val_cl, X_test_cl, y_test_cl,
     X_train_rg, y_train_rg, X_val_rg, y_val_rg, X_test_rg, y_test_rg,
     X_train_cl_scaled, X_val_cl_scaled, X_test_cl_scaled) = preparer.prepare_features(
        train_df, val_df, test_df
    )
    
    # Train models
    trainer = ModelTrainer(config)
    trainer.train_classifiers(X_train_cl, y_train_cl, X_train_cl_scaled)
    trainer.train_regressors(X_train_rg, y_train_rg)
    trainer.save_models(config, preparer.scaler)
    
    # Evaluate
    print("\n📊 Evaluating models...")
    evaluator = Evaluator()
    
    # Classification evaluation
    clf = trainer.models['tree_clf']
    proba_val = clf.predict_proba(X_val_cl)[:, 1]
    pred_val = (proba_val >= 0.5).astype(int)
    clf_metrics = evaluator.classification_metrics(y_val_cl, pred_val, proba_val)
    
    print(f"\n🎯 Classification (Validation):")
    for metric, value in clf_metrics.items():
        print(f"  {metric}: {value:.4f}")
    
    prec_at_k = evaluator.precision_at_k(y_val_cl, proba_val, config.TOP_K_PERCENT)
    print(f"  precision@{int(config.TOP_K_PERCENT*100)}%: {prec_at_k:.4f}")
    
    # Regression evaluation
    reg = trainer.models['tree_reg']
    pred_val_rg = reg.predict(X_val_rg)
    reg_metrics = evaluator.regression_metrics(y_val_rg, pred_val_rg)
    
    print(f"\n💰 Regression (Validation):")
    for metric, value in reg_metrics.items():
        print(f"  {metric}: {value:.4f}")
    
    # Plots
    evaluator.plot_roc_curve(y_val_cl, proba_val, config.FIG_DIR / "roc_curve_val.png")
    evaluator.plot_confusion_matrix(y_val_cl, pred_val, config.FIG_DIR / "confusion_matrix_val.png")
    
   # Business simulation
    print("\n💼 Running business simulation...")
    # Use the cleaned test dataframe stored in preparer
    test_df_clean = preparer.test_df_clean.reset_index(drop=True)
    proba_test_sim = clf.predict_proba(X_test_cl)[:, 1]
    
    test_df_sim = pd.DataFrame({
        'order_id': test_df_clean['order_id'].values if 'order_id' in test_df_clean.columns else range(len(proba_test_sim)),
        'order_value': test_df_clean['order_value'].values,
        'pred_proba': proba_test_sim,
        'true_leak': y_test_cl
    })
    
    simulator = BusinessSimulator(config)
    sim_df = simulator.run_simulation(test_df_sim)
    sim_df.to_csv(config.OUTPUTS / "business_simulation.csv", index=False)
    
    # Final test evaluation
    print("\n🧪 Final Test Set Evaluation:")
    proba_test = clf.predict_proba(X_test_cl)[:, 1]
    pred_test = (proba_test >= 0.5).astype(int)
    test_clf_metrics = evaluator.classification_metrics(y_test_cl, pred_test, proba_test)
    
    print(f"🎯 Classification (Test):")
    for metric, value in test_clf_metrics.items():
        print(f"  {metric}: {value:.4f}")
    
    pred_test_rg = reg.predict(X_test_rg)
    test_reg_metrics = evaluator.regression_metrics(y_test_rg, pred_test_rg)
    
    print(f"💰 Regression (Test):")
    for metric, value in test_reg_metrics.items():
        print(f"  {metric}: {value:.4f}")
    
# Score full dataset for dashboard
    print("\n📈 Scoring full dataset...")
    # Create a copy and add payment dummies if needed
    odf_scoring = odf.copy()

    # Add payment dummies to match training features
    if 'payment_type' in odf_scoring.columns and len(preparer.payment_dummies_cols) > 0:
        payment_dummies = pd.get_dummies(odf_scoring['payment_type'], prefix='pay', drop_first=True)
        # Ensure all training dummy columns exist
        for col in preparer.payment_dummies_cols:
            if col not in payment_dummies.columns:
                payment_dummies[col] = 0
        odf_scoring = pd.concat([odf_scoring.reset_index(drop=True), payment_dummies[preparer.payment_dummies_cols]], axis=1)

    # Ensure all feature columns exist and fill missing
    for col in preparer.feature_cols:
        if col not in odf_scoring.columns:
            odf_scoring[col] = 0

    full_features = odf_scoring[preparer.feature_cols].fillna(0).values
    odf['pred_leak_proba'] = clf.predict_proba(full_features)[:, 1]
    odf['pred_loss_amount'] = reg.predict(full_features)
    
    # Export top risky orders
    top_risky = odf.nlargest(200, 'pred_leak_proba')
    top_risky.to_csv(config.OUTPUTS / "top200_risky_orders.csv", index=False)
    
    # Save metadata
    metadata = {
        'features': preparer.feature_cols,
        'val_classification': clf_metrics,
        'val_regression': reg_metrics,
        'test_classification': test_clf_metrics,
        'test_regression': test_reg_metrics,
        'train_size': len(train_df),
        'val_size': len(val_df),
        'test_size': len(test_df)
    }
    
    pd.DataFrame([metadata]).to_json(config.OUTPUTS / "model_metadata.json", orient='records', indent=2)
    
    print("\n" + "=" * 80)
    print("✅ PIPELINE COMPLETE!")
    print("=" * 80)
    print(f"\n📁 Outputs saved to: {config.OUTPUTS}")
    print(f"  - Models: {len(list(config.MODELS_DIR.glob('*')))} files")
    print(f"  - Figures: {len(list(config.FIG_DIR.glob('*')))} files")
    print(f"  - Reports: business_simulation.csv, top200_risky_orders.csv")
    
    return odf, trainer.models, metadata


if __name__ == "__main__":
    odf, models, metadata = main()

REVENUE LEAKAGE DETECTION - OPTIMIZED PIPELINE
✓ Directories ready: C:\Users\naman\OneDrive\Desktop\retail revenue leakage detector\outputs

📥 Loading data...
✓ Loaded: olist_cleaned.csv | Shape: (119143, 43) | Memory: 176.2 MB

🔧 Engineering features...
✓ Using timestamp: order_purchase_timestamp
✓ Order-level aggregation: (99441, 12)
✓ Feature engineering complete: 17 columns

✂️ Splitting data...
✓ Time-based split: Train=69608, Val=14916, Test=14917

🎯 Preparing features...
✓ Features prepared: 9 features

🔹 Training classifiers...
✓ Classifiers trained

🔹 Training regressors...
✓ Regressors trained
✓ Saved 4 models + scaler

📊 Evaluating models...

🎯 Classification (Validation):
  roc_auc: 0.6704
  precision: 0.0019
  recall: 0.4375
  f1: 0.0037
  precision@5%: 0.0040

💰 Regression (Validation):
  mae: 0.6349
  rmse: 20.3216
  r2: -0.5271

💼 Running business simulation...

📊 Business Simulation Results:
 threshold  flagged  precision            net
       0.1    15257   0.010094 -